# Credit Card Fraud Detection Notebook Using Machine Learning  
This notebook includes the following components:  
- Setup  
- Visualization  
- Model training  
- Evaluation and comparison

## 1. Setup
It Includes:
* Importing Packages
* Seaborn for Figures
* Data Preprocessing Helper
* Pie and Bar Plot Helper
* Confusion Matrix Plotter
* Classifier Evaluation Helper


### 1.1. Importing Packages
It imports the libraries used across the notebook and defines the shared preprocessing and plotting helpers.
Here all necessary Python libraries for data manipulation, visualization, and machine learning are imported.

In [ ]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, mean_squared_error, r2_score, precision_score, recall_score, f1_score)
# from google.colab import drive

# drive.mount('/content/drive')

### 1.2. Using Seaborn for Figures
Here the global plotting parameters are configured using Seaborn for consistent visualization styles.

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

### 1.3. Data Preprocessing Helper
This function performs feature engineering and preprocessing steps on the input DataFrame. It drops unnecessary columns, converts date columns, extracts age, month, and year features, encodes gender, calculates distance features, and one-hot encodes the 'category' column.

In [ ]:
def data_pre(X: pd.DataFrame) -> pd.DataFrame:
    """Feature engineering and preprocessing used in the uploaded scripts."""
    X = X.copy()

    del_col = [
        "merchant", "first", "last", "street", "zip", "unix_time",
        "Unnamed: 0", "trans_num", "cc_num"
    ]
    X = X.drop(columns=del_col, errors="ignore")

    X["trans_date_trans_time"] = pd.to_datetime(X["trans_date_trans_time"])
    X["dob"] = pd.to_datetime(X["dob"])

    X["trans_date"] = pd.to_datetime(X["trans_date_trans_time"].dt.date)
    X["age"] = (X["trans_date"] - X["dob"]).dt.days // 365
    X["trans_month"] = X["trans_date"].dt.month
    X["trans_year"] = X["trans_date"].dt.year

    X["gender"] = X["gender"].map({"M": 1, "F": 0})

    X["lat_dis"] = (X["lat"] - X["merch_lat"]).abs()
    X["long_dis"] = (X["long"] - X["merch_long"]).abs()

    X = pd.get_dummies(X, columns=["category"], drop_first=True)

    X = X.drop(
        columns=[
            "city", "state", "job",
            "merch_lat", "merch_long",
            "lat", "long", "dob",
            "trans_date_trans_time", "trans_date"
        ],
        errors="ignore"
    )

    return X

### 1.4. Pie and Bar Plot Helper
This function generates a pie chart and a count plot for a given categorical column in a DataFrame. It helps visualize the distribution of categories and their relation to fraud status.

In [ ]:
def pie_bar_plot(df: pd.DataFrame, col: str) -> None:
    """Pie chart + count plot helper used for categorical EDA."""
    print(f"Value counts for {col}:")
    print(df[col].value_counts())

    fig, axs = plt.subplots(1, 2, figsize=(15, 6))
    counts = df[col].value_counts()

    axs[0].pie(
        counts.values.tolist(),
        autopct="%.2f%%",
        textprops={"fontsize": 12},
        shadow=True,
    )
    axs[0].set_title(f"{col} distribution")

    sns.countplot(data=df, x=col, hue="is_fraud", palette=["blue", "orange"], ax=axs[1])
    axs[1].set_title(f"{col} by fraud status")
    axs[1].legend(title="is_fraud", labels=["Not Fraud", "Fraud"])

    fig.legend(labels=counts.index.tolist(), loc="upper left")
    fig.tight_layout()
    plt.show()

### 1.5. Confusion Matrix Plotter
This helper function visualizes the confusion matrix, which is crucial for evaluating the performance of a classification model, especially in imbalanced datasets like fraud detection.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

### 1.6. ROC Curve Plotter

In [ ]:
def plot_roc_curve(y_true, y_pred_proba, model_name, roc_auc):
    plt.figure(figsize=(7, 5))
    plt.plot([0, 1], [0, 1], linestyle="--")
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.4f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} - Receiver Operating Characteristic (ROC) Curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

### 1.6. Classifier Evaluation Helper
This function provides a comprehensive evaluation of a given classifier model, including accuracy, ROC AUC, and a detailed classification report, along with a confusion matrix plot.

In [ ]:
def evaluate_classifier(name: str, model, X_test, y_test):
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, digits=4, output_dict=True, zero_division=0)
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]) if hasattr(model, "predict_proba") else np.nan,
        "precision_fraud": report["1"]["precision"],
        "recall_fraud": report["1"]["recall"],
        "f1_fraud": report["1"]["f1-score"],
    }
    print(f"=== {name} ===")
    print(f"Accuracy: {metrics['accuracy']:.2%}")
    if not np.isnan(metrics["roc_auc"]):
        print("ROC AUC:", metrics["roc_auc"])
    print(classification_report(y_test, y_pred, digits=4))
    plot_confusion_matrix(y_test, y_pred, f"{name} - Confusion Matrix")
    plot_roc_curve(y_test, model.predict_proba(X_test)[:, 1], name, metrics["roc_auc"])
    return metrics

## 2. Load the data
Upload `fraudTrain.csv` and `fraudTest.csv` in Colab, or place them in the working directory first.

In [ ]:
# train_path = "/content/drive/MyDrive/Dissertation_Files/fraudTrain.csv"
# test_path = "/content/drive/MyDrive/Dissertation_Files/fraudTest.csv"

train_path = "Datasets/fraudTrain.csv"
test_path = "Datasets/fraudTest.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

### 2.1. Training and Testing shape

In [ ]:
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

### 2.2. Training Data Overview

In [ ]:
display(train_df.head())

### 2.3. Testing Data Overview

In [ ]:
display(test_df.head())

### 2.4. Training Info

In [ ]:
train_df.info()

### 2.5.Target Distribution

In [ ]:
display(train_df["is_fraud"].value_counts().rename("count"))

## 3) Visual exploration
These plots come from the uploaded visualization script and helper plot module.

### 3.1 Bar Plot

In [ ]:
df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

pie_bar_plot(df, "is_fraud")
pie_bar_plot(df, "gender")

### 3.2. Log Scale

In [ ]:
fraud = df[df["is_fraud"] == 1]
non_fraud = df[df["is_fraud"] == 0].sample(min(5000, (df["is_fraud"] == 0).sum()), random_state=42)
df_sample = pd.concat([fraud, non_fraud], axis=0)
plt.figure(figsize=(12, 6))
sns.histplot(data=df_sample, x="amt", hue="is_fraud", bins=100, log_scale=True, alpha=0.6, multiple="stack")
plt.title("Transaction Amount Distribution by Fraud Status (Log Scale)")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 4) Preprocessing
The preprocessing step drops unneeded identifiers, extracts age and transaction date features, encodes gender, adds latitude/longitude distance features, and one-hot encodes transaction category. fileciteturn0file7

In [ ]:
train_pre = data_pre(train_df)
test_pre = data_pre(test_df)

X_train = train_pre.drop(columns=["is_fraud"])
y_train = train_pre["is_fraud"]

X_test = test_pre.drop(columns=["is_fraud"])
y_test = test_pre["is_fraud"]

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Preprocessed training shape:", X_train.shape)
print("Preprocessed testing shape:", X_test.shape)
print("Feature columns aligned:", list(X_train.columns) == list(X_test.columns))

## 5) Models
We train four classification models and compare them on the held-out test set.

Initialize a list to store results and a dictionary for fitted models.

In [ ]:
results = []
trained_models = {}

### 5.1. Logistic Regression

In [ ]:
name = "Logistic Regression"
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start
trained_models[name] = model
model_metrics = evaluate_classifier(name, model, X_test, y_test)
model_metrics["train_time_sec"] = elapsed
results.append(model_metrics)
print(f"\n{name} trained in {elapsed:.2f} sec")

### 5.2. SGD Classifier

In [ ]:
name = "SGD Classifier"
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42, class_weight='balanced'))
])
start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start
trained_models[name] = model
model_metrics = evaluate_classifier(name, model, X_test, y_test)
model_metrics["train_time_sec"] = elapsed
results.append(model_metrics)
print(f"\n{name} trained in {elapsed:.2f} sec")

### 5.3. XGBoost

In [ ]:
name = "XGBoost"
model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight= (len(y_train) - y_train.sum()) / y_train.sum()
)
start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start
trained_models[name] = model
model_metrics = evaluate_classifier(name, model, X_test, y_test)
model_metrics["train_time_sec"] = elapsed
results.append(model_metrics)
print(f"\n{name} trained in {elapsed:.2f} sec")

### 5.4. LightGBM

In [ ]:
name = "LightGBM"
model = lgb.LGBMClassifier(
    objective='binary',
    random_state=42,
    n_jobs=-1,
    scale_pos_weight= (len(y_train) - y_train.sum()) / y_train.sum()
)
start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start
trained_models[name] = model
model_metrics = evaluate_classifier(name, model, X_test, y_test)
model_metrics["train_time_sec"] = elapsed
results.append(model_metrics)
print(f"\n{name} trained in {elapsed:.2f} sec")

### Final Results DataFrame

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="f1_fraud", ascending=False).reset_index(drop=True)
display(results_df)

### Fraud Class F1 Score Comparison

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="model", y="f1_fraud")
plt.title("Fraud Class F1 Score Comparison")
plt.xlabel("Model")
plt.ylabel("F1 Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### ROC AUC Comparison

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="model", y="roc_auc")
plt.title("ROC AUC Comparison")
plt.xlabel("Model")
plt.ylabel("ROC AUC")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot([0, 1], [0, 1], linestyle="--")
for index, row in results_df.iterrows():
    name = row['model']
    model_instance = trained_models[name]
    y_pred_proba = model_instance.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {row['roc_auc']:.4f})")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison - All Models')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

## 6) Optional baseline: Linear Regression
This is not the best choice for a classification problem, but it is included because it appeared in the uploaded files. It can be used only as a rough baseline. fileciteturn0file4

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_pred_lin = lin_model.predict(X_test)
y_pred_lin_class = (y_pred_lin > 0.5).astype(int)

print("MSE:", mean_squared_error(y_test, y_pred_lin))
print("R^2:", r2_score(y_test, y_pred_lin))
print(classification_report(y_test, y_pred_lin_class, digits=4, zero_division=0))
plot_confusion_matrix(y_test, y_pred_lin_class, "Linear Regression - Confusion Matrix")

fpr, tpr, _ = roc_curve(y_test, y_pred_lin)
roc_auc = roc_auc_score(y_test, y_pred_lin)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Linear Regression ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

## 7) Final takeaway
By using the comparison table above to pick the model with the best fraud-class F1 score and ROC AUC. For this kind of imbalanced fraud problem, those metrics are usually more informative than accuracy alone.

### Based on the comparison table:
Best Model: XGBoost

Metrics: F1-score (Fraud) = 0.3171, ROC AUC = 0.9944

### Recommendation: Use XGBoost for deployment as it provides the best balance between precision and recall for the fraud class.
